# Middle mile: models M1, M2, M3

Delhivery public trip data, trips created 2018-09-12 to 2018-10-03. Planned time (`osrm_time`) is badly wrong: the median trip takes about 2.1x its planned time, so schedules and costs built on the plan are wrong.

| Task | Business question | Grain | Type |
|---|---|---|---|
| **M1** | How long will this trip really take? | trip | regression |
| **M2** | Which trips will be severely late (actual / planned >= 2.5)? | trip | classification |
| **M3** | How many minutes does a leg lose at hubs, and which centres cause it? | leg | regression |

This notebook contains **M1**, **M2** and **M3**.

In [1]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, joblib
from IPython.display import display, Markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from src import middle_mile as mm, evaluate as ev, vendor_pipeline as vp, config

## Build legs and trips

The time fields are **cumulative across checkpoint rows**, so they are handled carefully:

* **Leg** = one (`trip_uuid`, `source_center`, `destination_center`). `actual_time`, `osrm_time`, `osrm_distance` are cumulative, so the leg value is the **max**. `start_scan_to_end_scan`, `od_start_time`, `od_end_time` are constant per leg, so the first value is taken.
* **Trip** = `trip_uuid`. Leg values are **summed**. The first leg's source centre and the last leg's destination centre are kept (legs are ordered by `od_start_time`).
* **State** is parsed from the centre name (`City_Place_X (State)`); `corridor` = first source state -> last destination state.
* The file's own `data` column (training/test) is ignored: it is not a time split.

In [2]:
raw = mm.load_raw()
legs = mm.build_legs(raw)
trips = mm.build_trips(legs)
print(f"checkpoint rows {len(raw):,} | legs {len(legs):,} | trips {len(trips):,}")
print("trip_creation_time:", trips.trip_creation_time.min(), "to", trips.trip_creation_time.max())
print(f"median actual / planned (osrm) = {trips.ratio.median():.2f}x")
print("trips with missing values in model columns:", int(trips[["osrm_time_total", "osrm_distance_total", "route_type", "n_legs", "first_source_center",
      "last_destination_center", "corridor", "route_schedule_uuid", "creation_hour", "creation_weekday"]].isna().any(axis=1).sum()))

checkpoint rows 144,867 | legs 26,368 | trips 14,817
trip_creation_time: 2018-09-12 00:00:16.535741 to 2018-10-03 23:59:42.701692
median actual / planned (osrm) = 2.10x
trips with missing values in model columns: 0


## Split and leakage rules

* **Train:** `trip_creation_time` <= 2018-09-26 23:59:59. **Test:** 2018-09-27 to 2018-10-03. Never a random split.
* **Validation slice** (21-26 September) is reserved for threshold selection in the classification task M2; M1 uses fixed hyper-parameters (no search).
* **Never features:** `actual_time, segment_actual_time, actual_distance_to_destination, factor, segment_factor, start_scan_to_end_scan, od_end_time, cutoff_factor, cutoff_timestamp, is_cutoff` (they contain or are derived from the outcome).
* Route and centre "history" comes only from training trips, through `TargetEncoder(cv=5)` inside the Ridge pipeline; levels with fewer than 20 training trips become `"OTHER"`.

In [3]:
print("trips train / val-slice / test:", int(trips.train.sum()), "/", int(trips.val.sum()), "/", int(trips.test.sum()))
print("route_type mix:", trips.route_type.value_counts().to_dict())
display(trips[["actual_min", "osrm_time_total", "osrm_distance_total", "n_legs", "ratio"]].describe().round(1))

trips train / val-slice / test: 10654 / 4153 / 4163
route_type mix: {'Carting': 8908, 'FTL': 5909}


,actual_min,osrm_time_total,osrm_distance_total,n_legs,ratio
count,14817.0,14817.0,14817.0,14817.0,14817.0
mean,357.1,162.1,205.1,1.8,2.6
std,561.4,272.3,370.8,1.2,2.3
min,9.0,6.0,9.1,1.0,0.5
25%,67.0,29.0,30.9,1.0,1.7
50%,149.0,60.0,65.9,1.0,2.1
75%,370.0,169.0,209.3,2.0,2.8
max,6265.0,2032.0,2840.1,8.0,77.4


# M1: Trip duration (regression, one row per trip)

**Business question.** How long will this trip really take? Costs and schedules built on the planned time are wrong, so a model of the real duration is the input to better plans.

* **Target:** actual trip minutes (sum of the legs' `actual_time`). Models train on `log1p(minutes)` and are scored in minutes after converting back.
* **Features:** planned time and distance, route type, number of legs, first source centre, last destination centre, corridor, route schedule, creation hour and weekday.
* **Naive baseline:** `osrm_time_total` x the median training ratio (actual / planned).

In [4]:
M1_CATS = ["route_type", "first_source_center", "last_destination_center", "corridor", "route_schedule_uuid"]
M1_NUMS = ["osrm_time_total", "osrm_distance_total", "n_legs", "creation_hour", "creation_weekday"]
tr, te = trips["train"], trips["test"]
res = vp.run_regression(trips, M1_CATS, M1_NUMS, "actual_min", tr, te)
ratio_med = float(trips.loc[tr, "ratio"].median())
naive = ev.reg_metrics(trips.loc[te, "actual_min"], trips.loc[te, "osrm_time_total"] * ratio_med)
plan_only = ev.reg_metrics(trips.loc[te, "actual_min"], trips.loc[te, "osrm_time_total"])
tab = pd.DataFrame({**res["table"], f"Naive (planned x {ratio_med:.2f})": naive, "Planned time as-is (no adjustment)": plan_only}).T
tab["fit_seconds"] = pd.Series(res["fit_s"])
display(tab)
pr = pd.Series(res["pred"]["Ridge"], index=trips.index[te]); act = trips.loc[te, "actual_min"]
print(f"Ridge diagnostic: median abs error {np.median(np.abs(pr - act)):.0f} min (RandomForest {np.median(np.abs(pd.Series(res['pred']['RandomForest'], index=act.index) - act)):.0f}); "
      f"largest prediction {pr.max():,.0f} min vs largest actual {act.max():,.0f}; trips predicted > 3x actual: {int((pr > 3 * act).sum())}")
print("MAE in hours:", {k: round(v["MAE"] / 60, 2) for k, v in {**res["table"], "Naive": naive, "Planned as-is": plan_only}.items()})

,MAE,RMSE,MAPE,R2,fit_seconds
Ridge,169.24,621.88,47.86,-0.1399,0.0
RandomForest,58.64,146.88,24.45,0.9364,0.9
LightGBM,57.33,144.04,24.19,0.9389,1.0
Naive (planned x 2.09),80.55,177.26,29.22,0.9074,NaN
Planned time as-is (no adjustment),190.90,377.08,52.92,0.5809,NaN


Ridge diagnostic: median abs error 43 min (RandomForest 23); largest prediction 16,195 min vs largest actual 6,265; trips predicted > 3x actual: 117
MAE in hours: {'Ridge': 2.82, 'RandomForest': 0.98, 'LightGBM': 0.96, 'Naive': 1.34, 'Planned as-is': 3.18}


In [5]:
winner, best, ok = vp.pick_winner({m: res["table"][m]["MAE"] for m in res["table"]}, higher_is_better=False)
reason = (f"{winner} chosen on MAE ({res['table'][winner]['MAE']:,.0f} min). Models within 2% of the best MAE: {ok}; "
          "when several are within 2% the simplest is preferred.")
print("Winner:", winner, "|", reason)
te_df = trips[te].copy()
te_df["predicted"] = res["pred"][winner]
# 80% interval: LightGBM quantiles 0.1 / 0.9 fitted on trips created before 21 Sep, split-conformal (CQR) calibrated on the 21-26 Sep validation slice
early_m, val_m = tr & (trips["trip_creation_time"] < mm.VAL_START), tr & (trips["trip_creation_time"] >= mm.VAL_START)
ci1 = vp.conformal_interval(trips, M1_CATS, M1_NUMS, "actual_min", early_m, val_m, te, point_pred=te_df["predicted"].to_numpy())
lo_m, hi_m = ci1["q_low"], ci1["q_high"]
te_df["pi_low"], te_df["pi_high"] = ci1["pi_low"], ci1["pi_high"]
cov = ci1["test_coverage_calibrated"]
print(f"calibration trips (validation, 21-26 Sep): {ci1['n_cal']:,} | raw interval covers {ci1['val_coverage_raw']:.1%} of them; conformal correction Q = {ci1['Q']:.3f} on the log scale")
print(f"TEST coverage of the nominal 80% interval: raw {ci1['test_coverage_raw']:.1%} -> calibrated {cov:.1%} | mean width {ci1['test_mean_width_raw']:,.0f} -> {ci1['test_mean_width_calibrated']:,.0f} min")
pd.DataFrame({"record_id": te_df["trip_uuid"], "actual": te_df["actual_min"], "predicted": te_df["predicted"],
              "abs_error": (te_df["actual_min"] - te_df["predicted"]).abs(), "pi_low": te_df["pi_low"], "pi_high": te_df["pi_high"]}
             ).to_csv("../data/processed/pred_M1.csv", index=False)
print("saved pred_M1.csv", te_df.shape[0], "rows")

Winner: LightGBM | LightGBM chosen on MAE (57 min). Models within 2% of the best MAE: ['LightGBM']; when several are within 2% the simplest is preferred.


calibration trips (validation, 21-26 Sep): 4,153 | raw interval covers 64.0% of them; conformal correction Q = 0.078 on the log scale
TEST coverage of the nominal 80% interval: raw 64.5% -> calibrated 78.6% | mean width 130 -> 181 min
saved pred_M1.csv 4163 rows


### Business impact: hours of under-planned time, in ₹

A trip is **under-planned** by `max(actual - planned, 0)` minutes: the time the schedule did not budget, which has to be paid for as extra driver/vehicle hours. The same is computed against the **model's** predicted time. Cost per hour comes from the scenario rates in `src/config.py` (FTL and Carting), which are **sourced synthetic** rates, not Delhivery actuals. Over-planning (planned > actual) is not costed here.

In [6]:
rows = {}
for scen, rates in config.RATES_PER_HOUR.items():
    rate = te_df["route_type"].map(rates)
    plan_h = (te_df["actual_min"] - te_df["osrm_time_total"]).clip(lower=0) / 60
    model_h = (te_df["actual_min"] - te_df["predicted"]).clip(lower=0) / 60
    naive_h = (te_df["actual_min"] - te_df["osrm_time_total"] * ratio_med).clip(lower=0) / 60
    rows[scen] = {"rate_FTL": rates["FTL"], "rate_Carting": rates["Carting"],
                  "under_planned_hours_plan": plan_h.sum(), "under_planned_hours_model": model_h.sum(),
                  "under_planned_hours_naive": naive_h.sum(),
                  "Rs_total_plan": (plan_h * rate).sum(), "Rs_total_model": (model_h * rate).sum(), "Rs_total_naive": (naive_h * rate).sum(),
                  "Rs_per_trip_plan": (plan_h * rate).mean(), "Rs_per_trip_model": (model_h * rate).mean(), "Rs_per_trip_naive": (naive_h * rate).mean()}
impact = pd.DataFrame(rows).T
impact["reduction_vs_plan_%"] = (1 - impact["Rs_total_model"] / impact["Rs_total_plan"]) * 100
impact["reduction_vs_naive_%"] = (1 - impact["Rs_total_model"] / impact["Rs_total_naive"]) * 100
display(impact[["under_planned_hours_plan", "under_planned_hours_naive", "under_planned_hours_model", "Rs_total_plan", "Rs_total_naive", "Rs_total_model", "Rs_per_trip_plan", "Rs_per_trip_naive", "Rs_per_trip_model", "reduction_vs_plan_%", "reduction_vs_naive_%"]].round(1))
print(f"test trips: {len(te_df):,} | share of trips under-planned by the plan: {(te_df.actual_min > te_df.osrm_time_total).mean():.1%}; "
      f"by the model: {(te_df.actual_min > te_df.predicted).mean():.1%}")
print("rates are", config.RATE_SOURCE)

,under_planned_hours_plan,under_planned_hours_naive,under_planned_hours_model,Rs_total_plan,Rs_total_naive,Rs_total_model,Rs_per_trip_plan,Rs_per_trip_naive,Rs_per_trip_model,reduction_vs_plan_%,reduction_vs_naive_%
base,13242.0,3535.5,2386.6,5939905.1,1259946.7,918820.7,1426.8,302.7,220.7,84.5,27.1
conservative,13242.0,3535.5,2386.6,5184300.8,1080431.3,792982.2,1245.3,259.5,190.5,84.7,26.6
optimistic,13242.0,3535.5,2386.6,6695509.3,1439462.1,1044659.2,1608.3,345.8,250.9,84.4,27.4


test trips: 4,163 | share of trips under-planned by the plan: 99.3%; by the model: 41.1%
rates are sourced synthetic


In [7]:
tree = winner if winner != "Ridge" else min(["RandomForest", "LightGBM"], key=lambda m: res["table"][m]["MAE"])
print("SHAP model:", tree)
drivers, _, imp = vp.shap_report(res["fitted"][tree], trips[tr][M1_CATS + M1_NUMS], M1_CATS, "M1",
    "Planned time and distance, plus the route, explain most of the real trip duration",
    "What drives the predicted trip duration (SHAP, log minutes)", "trip duration (log minutes)",
    sample=2000 if tree == "LightGBM" else 400)
if tree != winner:
    print(f"Note: SHAP drivers come from {tree} because the winner ({winner}) is not a tree model.")
display(pd.DataFrame(drivers))

SHAP model: LightGBM


,feature,mean_abs_shap,plain_english
0,osrm_distance_total,0.65540,Higher osrm_distance_total raises the predicte...
1,osrm_time_total,0.21469,Higher osrm_time_total raises the predicted tr...
2,creation_hour,0.07027,Higher creation_hour lowers the predicted trip...
3,corridor,0.06608,corridor: the biggest push toward trip duratio...
4,last_destination_center,0.06365,last_destination_center: the biggest push towa...


In [8]:
from src.middle_mile import NEVER_FEATURES
joblib.dump({"pipeline": res["fitted"][winner], "q_low": lo_m, "q_high": hi_m, "conformal_Q": ci1["Q"], "cats": M1_CATS, "nums": M1_NUMS}, f"../models/M1_{winner}.joblib", compress=3)
vp.save_json("M1", {
    "task_id": "M1", "problem": "middle_mile", "type": "regression",
    "business_question": "How long will this trip really take?",
    "target": "actual trip minutes (sum of legs' actual_time; models trained on log1p)", "grain": "trip",
    "train_period": "trips created 2018-09-12 to 2018-09-26", "test_period": "trips created 2018-09-27 to 2018-10-03",
    "n_train": int(tr.sum()), "n_test": int(te.sum()),
    "base_rate_or_baseline": {"metric": "osrm_time_total x median training ratio (MAE, minutes)", "value": naive["MAE"], "all_metrics": naive,
                              "median_training_ratio": ratio_med, "planned_time_as_is": plan_only},
    "features": M1_CATS + M1_NUMS, "excluded_for_leakage": NEVER_FEATURES,
    "models": [{"name": m, "metrics": res["table"][m], "fit_seconds": res["fit_s"][m]} for m in res["table"]],
    "winner": winner, "winner_reason": reason, "threshold": None, "top_drivers": drivers,
    "business_impact_inputs": {"label": "sourced synthetic rates; Rs of under-planned hours in the test period",
        "rates_per_hour": config.RATES_PER_HOUR, "n_test_trips": int(len(te_df)),
        "scenarios": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in impact.to_dict("index").items()},
        "interval_80pct_coverage": cov, "interval_coverage_before_calibration": ci1["test_coverage_raw"],
        "conformal": {k: ci1[k] for k in ("Q", "n_cal", "target_coverage", "val_coverage_raw", "test_coverage_raw", "test_coverage_calibrated", "test_mean_width_raw", "test_mean_width_calibrated")}},
    "caveats": ["Data is from 2018 and covers about 3 weeks (trips created 2018-09-12 to 2018-10-03).",
                "No weather or traffic data; the dataset has no cost fields (rates are sourced synthetic).",
                "Hyper-parameters are fixed defaults (no search); one temporal split (final week as test).",
                f"pi_low / pi_high: LightGBM quantiles calibrated by split-conformal (CQR) on the 21-26 Sep validation slice (Q = {ci1['Q']:.3f} on the log scale); test coverage of the nominal 80% interval is {cov:.1%} (raw quantiles {ci1['test_coverage_raw']:.1%}). Calibration used only 6 days, so coverage may not hold in other weeks.",
                "route_schedule_uuid and centre codes are high-cardinality; levels with < 20 training trips are grouped into OTHER."]})

# M2: Severe-delay risk (classification, one row per trip)

**Business question.** Which trips will be severely late, so that ops can re-plan, add buffer or reassign before dispatch? This is the explainability model for the middle-mile problem.

**Label.** `severe overrun` = actual / planned (`osrm`) time >= **2.5**. Why not 1.5x: almost every trip is above it, so that label tells us almost nothing. The table below shows the share of trips at each threshold.

In [9]:
THRESHOLDS = [1.5, 2.0, 2.5, 3.0]
shares = pd.DataFrame({"share of trips at or above (%)": [round((trips.ratio >= t).mean() * 100, 1) for t in THRESHOLDS]}, index=[f"{t}x" for t in THRESHOLDS])
display(shares)
trips["lr_date"] = trips["trip_creation_time"]
trips["severe"] = (trips["ratio"] >= 2.5).astype(int)
print(f"severe-overrun base rate: all {trips.severe.mean():.3f} | train {trips.loc[tr, 'severe'].mean():.3f} | test {trips.loc[te, 'severe'].mean():.3f}")

,share of trips at or above (%)
1.5x,88.8
2.0x,57.1
2.5x,32.2
3.0x,20.3


severe-overrun base rate: all 0.322 | train 0.318 | test 0.330


**Features** are the same as M1 (planned time and distance, route type, number of legs, first source centre, last destination centre, corridor, route schedule, creation hour and weekday). **Models:** Logistic (`class_weight="balanced"`), RandomForest, LightGBM; naive baseline = the base rate.

**Threshold** (AGENTS.md rule): the value that maximises recall while keeping precision >= 0.60 on the validation slice (21-26 September, scored by a copy of each model trained on 12-20 September); if 0.60 is not reachable, the best-F1 threshold. The reported model is trained on the full training period.

In [10]:
res2 = vp.run_classification(trips, M1_CATS, M1_NUMS, "severe", tr, te, mm.VAL_START)
y2 = trips.loc[te, "severe"].values
prev2 = round(float(y2.mean()), 4)
tab2 = pd.DataFrame(res2["table"]).T
tab2.loc["Base rate (constant)"] = {"PR_AUC": prev2}
# Lookup baseline: historical severe-overrun rate per route_schedule_uuid, fitted on TRAIN trips only (< 5 train trips -> train base rate)
from sklearn.metrics import average_precision_score
rate2, base2, _ = vp.lookup_rate(trips[tr], trips[te], ["route_schedule_uuid"], "severe")
lookup2 = round(float(average_precision_score(y2, rate2)), 4)
n_seen2 = int(trips.loc[te, "route_schedule_uuid"].isin(trips.loc[tr, "route_schedule_uuid"]).sum())
tab2.loc["Lookup (route_schedule rate, train only)"] = {"PR_AUC": lookup2}
print(f"lookup baseline PR-AUC {lookup2:.4f} (train base rate {base2:.3f}; {n_seen2:,} of {int(te.sum()):,} test trips have a schedule seen in training)")
tab2["fit_seconds"] = pd.Series(res2["fit_s"])
display(tab2)

lookup baseline PR-AUC 0.7098 (train base rate 0.318; 3,853 of 4,163 test trips have a schedule seen in training)


,PR_AUC,ROC_AUC,precision,recall,F1,threshold,val_precision_at_thr,val_recall_at_thr,thr_rule,fit_seconds
Logistic,0.6304,0.7862,0.6068,0.5758,0.5909,0.6148,0.6,0.5241,recall-max with val precision>=0.60,0.1
RandomForest,0.7521,0.8526,0.6041,0.7864,0.6833,0.4227,0.6001,0.747,recall-max with val precision>=0.60,0.9
LightGBM,0.7574,0.8526,0.6087,0.7813,0.6843,0.4709,0.6,0.7297,recall-max with val precision>=0.60,0.6
Base rate (constant),0.3296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"Lookup (route_schedule rate, train only)",0.7098,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
winner2, best2, ok2 = vp.pick_winner({m: res2["table"][m]["PR_AUC"] for m in res2["table"]}, higher_is_better=True)
reason2 = f"{winner2} chosen on PR-AUC ({res2['table'][winner2]['PR_AUC']:.4f}); models within 2% of the best: {ok2} (simplest preferred)."
thr2 = res2["thr"][winner2]; p2 = res2["proba"][winner2]
lab2 = (p2 >= thr2).astype(int)
print("Winner:", winner2, "|", reason2)
print(res2["table"][winner2]["thr_rule"], "| threshold =", round(thr2, 4), "| flagged:", int(lab2.sum()), "of", len(lab2))
m2 = trips[te].copy(); m2["flag"] = lab2
pd.DataFrame({"record_id": m2["trip_uuid"], "actual": m2["severe"], "predicted_label": lab2, "probability": p2, "threshold": thr2}
             ).to_csv("../data/processed/pred_M2.csv", index=False)
pc, rc, tc = ev.pr_curve(y2, p2)
pd.DataFrame({"precision": pc[:-1], "recall": rc[:-1], "threshold": tc}).to_csv("../reports/metrics/M2_pr_curve.csv", index=False)

Winner: RandomForest | RandomForest chosen on PR-AUC (0.7521); models within 2% of the best: ['RandomForest', 'LightGBM'] (simplest preferred).
recall-max with val precision>=0.60 | threshold = 0.4227 | flagged: 1786 of 4163


### Business impact (test period)

Excess hours on the correctly flagged trips = `max(actual - planned, 0)` hours, costed at the scenario rate for the trip's route type (`src/config.py`; **sourced synthetic** rates).

In [12]:
tp = m2[(m2.flag == 1) & (m2.severe == 1)]
exc_h = (tp["actual_min"] - tp["osrm_time_total"]).clip(lower=0) / 60
impact2 = {"label": "test period; rates are sourced synthetic", "flagged": int(lab2.sum()), "true_positives": int(len(tp)),
           "actual_severe_trips": int(m2.severe.sum()), "excess_hours_on_true_positives": float(exc_h.sum()),
           "Rs_on_true_positives": {sc: float((exc_h * tp["route_type"].map(r)).sum()) for sc, r in config.RATES_PER_HOUR.items()}}
print({k: (round(v, 1) if isinstance(v, float) else v) for k, v in impact2.items()})

{'label': 'test period; rates are sourced synthetic', 'flagged': 1786, 'true_positives': 1079, 'actual_severe_trips': 1372, 'excess_hours_on_true_positives': 3346.4, 'Rs_on_true_positives': {'base': 1064973.25, 'conservative': 903758.3333333334, 'optimistic': 1226188.1666666665}}


In [13]:
tree2 = winner2 if winner2 != "Logistic" else "LightGBM"
Xte2 = trips[te][M1_CATS + M1_NUMS]
drivers2, (Xs2, Z2, sv2, b2), imp2 = vp.shap_report(res2["fitted"][tree2], Xte2, M1_CATS, "M2",
    "Planned time and distance, and the corridor, decide which trips run severely late",
    "What pushes a trip toward a severe overrun (SHAP, log-odds)", "severe-overrun risk",
    sample=2000 if tree2 == "LightGBM" else 400)
display(pd.DataFrame(drivers2))
ps = res2["fitted"][tree2].predict_proba(Xs2)[:, 1]
idx = {"high_risk": int(np.argmax(ps)), "borderline": int(np.argmin(np.abs(ps - thr2))), "safe": int(np.argmin(ps))}
ex_rows = []
for k, i in idx.items():
    vp.waterfall_png(Z2, sv2, b2, i, f"../reports/figures/M2_waterfall_{k}.png", f"{k.replace('_', ' ').title()} trip: why the model scores it {ps[i]:.2f}")
    r = trips.loc[Xs2.index[i]]
    ex_rows.append({"case": k, "model_probability": round(float(ps[i]), 3), "route_type": r.route_type, "corridor": r.corridor,
                    "planned_min": round(r.osrm_time_total), "actual_min": round(r.actual_min), "actual_over_planned": round(r.ratio, 2), "severe": int(r.severe)})
display(pd.DataFrame(ex_rows))

,feature,mean_abs_shap,plain_english
0,osrm_time_total,0.08104,Higher osrm_time_total lowers the predicted se...
1,creation_hour,0.05930,Higher creation_hour lowers the predicted seve...
2,corridor,0.05060,corridor: the biggest push toward severe-overr...
3,first_source_center,0.05042,first_source_center: the biggest push toward s...
4,osrm_distance_total,0.04347,Higher osrm_distance_total lowers the predicte...


,case,model_probability,route_type,corridor,planned_min,actual_min,actual_over_planned,severe
0,high_risk,0.999,FTL,Bihar -> Bihar,183,1007,5.50,1
1,borderline,0.425,FTL,Haryana -> Punjab,211,393,1.86,0
2,safe,0.000,FTL,Haryana -> Karnataka,1548,3022,1.95,0


### M2 limitations
* The data is from 2018 and covers only about 3 weeks; the test week is the last week of the file and may not represent other seasons.
* There is no weather or traffic data, so delays caused by them can only show up through route and time-of-day patterns.
* The dataset has no cost fields: the ₹ figures use the scenario rates in `src/config.py`, which are sourced synthetic, not Delhivery actuals.
* Hyper-parameters are fixed defaults (no search), and thresholds were chosen on a 6-day validation slice.

In [14]:
joblib.dump({"pipeline": res2["fitted"][winner2], "threshold": thr2, "cats": M1_CATS, "nums": M1_NUMS}, f"../models/M2_{winner2}.joblib", compress=3)
vp.save_json("M2", {
    "task_id": "M2", "problem": "middle_mile", "type": "classification",
    "business_question": "Which trips will be severely late (actual / planned >= 2.5)?",
    "target": "actual_min / osrm_time_total >= 2.5", "grain": "trip",
    "train_period": "trips created 2018-09-12 to 2018-09-26", "test_period": "trips created 2018-09-27 to 2018-10-03",
    "n_train": int(tr.sum()), "n_test": int(te.sum()),
    "base_rate_or_baseline": {"metric": "PR-AUC of constant base rate (= test prevalence)", "value": prev2,
                              "train_base_rate": round(float(trips.loc[tr, "severe"].mean()), 4)},
    "extra_baselines": [{"name": "Lookup: severe-overrun rate by route_schedule_uuid, fitted on TRAIN trips only (<5 trips -> train base rate)", "PR_AUC": lookup2,
                         "test_trips_with_schedule_seen_in_train": n_seen2}],
    "features": M1_CATS + M1_NUMS, "excluded_for_leakage": NEVER_FEATURES,
    "models": [{"name": m, "metrics": {k: v for k, v in res2["table"][m].items() if k != "thr_rule"}, "fit_seconds": res2["fit_s"][m]} for m in res2["table"]],
    "winner": winner2, "winner_reason": reason2, "threshold": thr2, "threshold_rule": res2["table"][winner2]["thr_rule"],
    "top_drivers": drivers2, "business_impact_inputs": impact2,
    "overrun_share_table": {"share_pct": shares.iloc[:, 0].to_dict()},
    "caveats": ["Data is from 2018 and covers about 3 weeks.", "No weather or traffic data.", "No cost fields; rates are sourced synthetic.",
                "Hyper-parameters are fixed defaults; threshold chosen on a 6-day validation slice."]})

# M3: Hub handling time (regression, one row per leg)

**Business question.** How many minutes does a leg lose at hubs beyond driving time, and which centres cause it? The action is a hub-improvement list.

* **Target:** `handling_min = start_scan_to_end_scan - leg actual_time`, clipped at >= 0. Models train on `log1p(handling_min)` and are scored in minutes.
* **Features:** source centre, destination centre, route type, planned leg time and distance, leg position and number of legs in the trip, start hour and weekday, and `source_daily_leg_volume` (the number of legs starting at that centre that day, a congestion proxy; the schedule is known in advance).
* **Split:** by the trip's creation time, the same train / test cut as M1 and M2 (a leg stays on the side of its trip).
* **Naive baseline:** the training median handling time of the leg's source centre (global training median for centres never seen in training).
* **Leakage:** `start_scan_to_end_scan` and `actual_time` build the target only; neither is a feature, and no other forbidden column is used.

In [15]:
fmt_rs = lambda x: f"₹{x:,.0f}"
legs["handling_min"] = (legs["start_scan_to_end_scan"] - legs["actual_time"]).clip(lower=0)
legs["od_start_hour"] = legs["od_start_time"].dt.hour
legs["od_start_weekday"] = legs["od_start_time"].dt.weekday
legs["source_daily_leg_volume"] = legs.groupby(["source_center", legs["od_start_time"].dt.date])["trip_uuid"].transform("size")
legs["train"] = legs["trip_creation_time"] <= mm.TRAIN_END
legs["test"] = legs["trip_creation_time"] >= mm.TEST_START
h = legs["handling_min"]
print(f"legs {len(legs):,} | train {int(legs.train.sum()):,} | test {int(legs.test.sum()):,}")
print(f"handling_min: median {h.median():.0f}, p90 {h.quantile(.9):.0f}, share clipped to 0: {(h == 0).mean():.1%}")
print("source centres:", legs.source_center.nunique(), "| daily volume per source: median", int(legs.source_daily_leg_volume.median()), "max", int(legs.source_daily_leg_volume.max()))

legs 26,368 | train 18,947 | test 7,421
handling_min: median 49, p90 238, share clipped to 0: 0.6%
source centres: 1508 | daily volume per source: median 2 max 70


In [16]:
M3_CATS = ["source_center", "destination_center", "route_type"]
M3_NUMS = ["osrm_time", "osrm_distance", "leg_position", "legs_in_trip", "od_start_hour", "od_start_weekday", "source_daily_leg_volume"]
tr3, te3 = legs["train"], legs["test"]
res3 = vp.run_regression(legs, M3_CATS, M3_NUMS, "handling_min", tr3, te3)
med_by_src = legs.loc[tr3].groupby("source_center")["handling_min"].median()
glob_med = float(legs.loc[tr3, "handling_min"].median())
naive3 = ev.reg_metrics(legs.loc[te3, "handling_min"], legs.loc[te3, "source_center"].map(med_by_src).fillna(glob_med))
tab3 = pd.DataFrame({**res3["table"], "Naive (train median by source centre)": naive3}).T
tab3["fit_seconds"] = pd.Series(res3["fit_s"])
display(tab3)

,MAE,RMSE,MAPE,R2,fit_seconds
Ridge,58.30,138.85,90.17,0.1022,0.0
RandomForest,49.12,126.02,77.27,0.2604,1.7
LightGBM,47.52,122.70,78.00,0.2988,1.3
Naive (train median by source centre),53.75,129.94,95.32,0.2136,NaN


In [17]:
winner3, best3, ok3 = vp.pick_winner({m: res3["table"][m]["MAE"] for m in res3["table"]}, higher_is_better=False)
reason3 = (f"{winner3} chosen on MAE ({res3['table'][winner3]['MAE']:,.1f} min). Models within 2% of the best MAE: {ok3}; "
           "when several are within 2% the simplest is preferred.")
print("Winner:", winner3, "|", reason3)
te_l = legs[te3].copy()
te_l["predicted"] = res3["pred"][winner3]
early3, val3 = tr3 & (legs["trip_creation_time"] < mm.VAL_START), tr3 & (legs["trip_creation_time"] >= mm.VAL_START)
ci3 = vp.conformal_interval(legs, M3_CATS, M3_NUMS, "handling_min", early3, val3, te3, point_pred=te_l["predicted"].to_numpy())
lo3, hi3 = ci3["q_low"], ci3["q_high"]
te_l["pi_low"], te_l["pi_high"] = ci3["pi_low"], ci3["pi_high"]
cov3 = ci3["test_coverage_calibrated"]
print(f"calibration legs (validation, 21-26 Sep): {ci3['n_cal']:,} | raw interval covers {ci3['val_coverage_raw']:.1%} of them; conformal correction Q = {ci3['Q']:.3f} on the log scale")
print(f"TEST coverage of the nominal 80% interval: raw {ci3['test_coverage_raw']:.1%} -> calibrated {cov3:.1%} | mean width {ci3['test_mean_width_raw']:,.0f} -> {ci3['test_mean_width_calibrated']:,.0f} min")
te_l["record_id"] = te_l["trip_uuid"] + "|" + te_l["source_center"] + "|" + te_l["destination_center"]
pd.DataFrame({"record_id": te_l["record_id"], "actual": te_l["handling_min"], "predicted": te_l["predicted"],
              "abs_error": (te_l["handling_min"] - te_l["predicted"]).abs(), "pi_low": te_l["pi_low"], "pi_high": te_l["pi_high"]}
             ).to_csv("../data/processed/pred_M3.csv", index=False)
print("saved pred_M3.csv", len(te_l), "rows; record_id unique:", te_l["record_id"].is_unique)

Winner: LightGBM | LightGBM chosen on MAE (47.5 min). Models within 2% of the best MAE: ['LightGBM']; when several are within 2% the simplest is preferred.


calibration legs (validation, 21-26 Sep): 7,507 | raw interval covers 68.7% of them; conformal correction Q = 0.176 on the log scale
TEST coverage of the nominal 80% interval: raw 68.1% -> calibrated 79.7% | mean width 101 -> 132 min
saved pred_M3.csv 7421 rows; record_id unique: True


### Hub-improvement ranking

One row per **source centre with at least 30 test legs**. `excess_hours` = hours above the **network median handling time of the test legs** (each leg's excess is `max(handling - median, 0)`), costed at the base scenario rate for the leg's route type (FTL or Carting; **sourced synthetic** rates from `src/config.py`). The model's own mean prediction is shown next to the actual mean, and the check below tests whether the model would have picked the same hubs.

In [18]:
net_med = float(te_l["handling_min"].median())
te_l["rate"] = te_l["route_type"].map(config.RATES_PER_HOUR["base"])
te_l["excess_h"] = (te_l["handling_min"] - net_med).clip(lower=0) / 60
te_l["excess_h_pred"] = (te_l["predicted"] - net_med).clip(lower=0) / 60
hub = te_l.groupby("source_center").agg(source_name=("source_name", "first"), legs=("trip_uuid", "size"),
        mean_actual_handling_min=("handling_min", "mean"), mean_predicted_handling_min=("predicted", "mean"),
        excess_hours=("excess_h", "sum"), excess_hours_predicted=("excess_h_pred", "sum")).reset_index()
hub["excess_rs_base"] = te_l.assign(c=te_l["excess_h"] * te_l["rate"]).groupby("source_center")["c"].sum().reindex(hub["source_center"]).values
hub = hub[hub.legs >= 30].sort_values("excess_hours", ascending=False).reset_index(drop=True)
hub.to_csv("../data/processed/hub_handling_ranking.csv", index=False)
print(f"network median handling (test legs) = {net_med:.0f} min | hubs with >= 30 test legs: {len(hub)} of {te_l.source_center.nunique()}")
top = hub.head(10)
display(top[["source_center", "source_name", "legs", "mean_actual_handling_min", "mean_predicted_handling_min", "excess_hours", "excess_rs_base"]].round(1))
from scipy.stats import spearmanr
rho = spearmanr(hub["mean_actual_handling_min"], hub["mean_predicted_handling_min"])[0]
top_pred = set(hub.nlargest(10, "mean_predicted_handling_min")["source_center"])
overlap = len(set(top["source_center"]) & top_pred)
print(f"hub-level agreement, actual vs predicted mean handling: Spearman {rho:.2f}; top-10 overlap {overlap}/10")
print(f"total excess: {hub.excess_hours.sum():,.0f} h = {fmt_rs(hub.excess_rs_base.sum())} (base rate; hubs with >= 30 test legs); top 10 hubs hold {top.excess_hours.sum() / hub.excess_hours.sum():.0%} of it")

network median handling (test legs) = 51 min | hubs with >= 30 test legs: 25 of 1140


,source_center,source_name,legs,mean_actual_handling_min,mean_predicted_handling_min,excess_hours,excess_rs_base
0,IND000000ACB,Gurgaon_Bilaspur_HB (Haryana),286,188.6,149.2,658.9,281047.7
1,IND421302AAG,Bhiwandi_Mankoli_HB (Maharashtra),267,168.7,146.0,530.9,158342.4
2,IND562132AAA,Bangalore_Nelmngla_H (Karnataka),194,169.6,147.2,391.6,132433.8
3,IND501359AAE,Hyderabad_Shamshbd_H (Telangana),78,216.4,161.0,215.0,90636.2
4,IND209304AAA,Kanpur_Central_H_6 (Uttar Pradesh),82,196.1,140.5,200.9,101371.8
5,IND382430AAB,Ahmedabad_East_H_1 (Gujarat),77,191.9,131.9,183.6,85086.4
6,IND160002AAC,Chandigarh_Mehmdpur_H (Punjab),74,173.2,134.3,152.6,82154.2
7,IND411033AAA,Pune_Tathawde_H (Maharashtra),84,152.1,87.5,148.4,68521.4
8,IND110037AAM,Delhi_Airport_H (Delhi),71,170.6,135.4,146.1,67219.8
9,IND131028AAB,Sonipat_Kundli_H (Haryana),45,213.1,155.0,122.1,59614.5


hub-level agreement, actual vs predicted mean handling: Spearman 0.89; top-10 overlap 8/10
total excess: 3,536 h = ₹1,399,642 (base rate; hubs with >= 30 test legs); top 10 hubs hold 78% of it


In [19]:
tree3 = winner3 if winner3 != "Ridge" else min(["RandomForest", "LightGBM"], key=lambda m: res3["table"][m]["MAE"])
print("SHAP model:", tree3)
drivers3, _, imp3 = vp.shap_report(res3["fitted"][tree3], legs[tr3][M3_CATS + M3_NUMS], M3_CATS, "M3",
    "Which hub a leg starts from, and how busy it is, explain the time lost at hubs",
    "What drives predicted hub handling time (SHAP, log minutes)", "hub handling time (log minutes)",
    sample=2000 if tree3 == "LightGBM" else 400)
if tree3 != winner3:
    print(f"Note: SHAP drivers come from {tree3} because the winner ({winner3}) is not a tree model.")
display(pd.DataFrame(drivers3))

SHAP model: LightGBM


,feature,mean_abs_shap,plain_english
0,source_center,0.46033,source_center: the biggest push toward hub han...
1,destination_center,0.24218,destination_center: the biggest push toward hu...
2,od_start_hour,0.15327,Higher od_start_hour raises the predicted hub ...
3,osrm_distance,0.10240,Higher osrm_distance raises the predicted hub ...
4,source_daily_leg_volume,0.07895,Higher source_daily_leg_volume raises the pred...


In [20]:
joblib.dump({"pipeline": res3["fitted"][winner3], "q_low": lo3, "q_high": hi3, "conformal_Q": ci3["Q"], "cats": M3_CATS, "nums": M3_NUMS}, f"../models/M3_{winner3}.joblib", compress=3)
vp.save_json("M3", {
    "task_id": "M3", "problem": "middle_mile", "type": "regression",
    "business_question": "How many minutes does a leg lose at hubs beyond driving time, and which centres cause it?",
    "target": "handling_min = max(start_scan_to_end_scan - leg actual_time, 0); models trained on log1p", "grain": "leg",
    "train_period": "legs of trips created 2018-09-12 to 2018-09-26", "test_period": "legs of trips created 2018-09-27 to 2018-10-03",
    "n_train": int(tr3.sum()), "n_test": int(te3.sum()),
    "base_rate_or_baseline": {"metric": "training median handling by source centre (MAE, minutes)", "value": naive3["MAE"], "all_metrics": naive3,
                              "global_median_fallback": glob_med},
    "features": M3_CATS + M3_NUMS, "excluded_for_leakage": NEVER_FEATURES,
    "models": [{"name": m, "metrics": res3["table"][m], "fit_seconds": res3["fit_s"][m]} for m in res3["table"]],
    "winner": winner3, "winner_reason": reason3, "threshold": None, "top_drivers": drivers3,
    "business_impact_inputs": {"label": "sourced synthetic rates; base scenario", "network_median_handling_min_test": net_med,
        "hubs_with_30plus_test_legs": int(len(hub)), "total_excess_hours": float(hub.excess_hours.sum()),
        "total_excess_rs_base": float(hub.excess_rs_base.sum()), "top10_share_of_excess": float(top.excess_hours.sum() / hub.excess_hours.sum()),
        "hub_level_spearman_actual_vs_predicted": float(rho), "top10_overlap_actual_vs_predicted": int(overlap),
        "interval_80pct_coverage": cov3, "interval_coverage_before_calibration": ci3["test_coverage_raw"],
        "conformal": {k: ci3[k] for k in ("Q", "n_cal", "target_coverage", "val_coverage_raw", "test_coverage_raw", "test_coverage_calibrated", "test_mean_width_raw", "test_mean_width_calibrated")}},
    "caveats": ["Data is from 2018 and covers about 3 weeks; one test week.", "No weather, traffic or hub-staffing data.",
                "No cost fields; the rupee figures use sourced synthetic rates.",
                "source_daily_leg_volume counts legs in this extract only; it is a congestion proxy, not true hub load.",
                "Handling time is derived as scan-to-scan time minus recorded driving time, so recording errors in either appear as handling time.",
                f"pi_low / pi_high: LightGBM quantiles calibrated by split-conformal (CQR) on the 21-26 Sep validation slice (Q = {ci3['Q']:.3f} on the log scale); test coverage of the nominal 80% interval is {cov3:.1%} (raw quantiles {ci3['test_coverage_raw']:.1%}).",
                "Hyper-parameters are fixed defaults (no search)."]})

## Save the trip and leg tables (used by the app)

In [21]:
trips.drop(columns=["lr_date"]).to_parquet("../data/processed/middle_mile_trips.parquet")
legs.to_parquet("../data/processed/middle_mile_legs.parquet")
print("saved middle_mile_trips.parquet", trips.shape, "and middle_mile_legs.parquet", legs.shape)

saved middle_mile_trips.parquet (14817, 21) and middle_mile_legs.parquet (26368, 24)


## Checkpoint summary (M1, M2, M3)

In [22]:
import json
for t in ["M1", "M2", "M3"]:
    d = json.load(open(f"../reports/metrics/{t}.json", encoding="utf8"))
    print(f"\n=== {t}: {d['business_question']}  (winner: {d['winner']}, n_train={d['n_train']:,}, n_test={d['n_test']:,})")
    display(pd.DataFrame({m["name"]: m["metrics"] for m in d["models"]}).T)


=== M1: How long will this trip really take?  (winner: LightGBM, n_train=10,654, n_test=4,163)


,MAE,RMSE,MAPE,R2
Ridge,169.24,621.88,47.86,-0.1399
RandomForest,58.64,146.88,24.45,0.9364
LightGBM,57.33,144.04,24.19,0.9389



=== M2: Which trips will be severely late (actual / planned >= 2.5)?  (winner: RandomForest, n_train=10,654, n_test=4,163)


,PR_AUC,ROC_AUC,precision,recall,F1,threshold,val_precision_at_thr,val_recall_at_thr
Logistic,0.6304,0.7862,0.6068,0.5758,0.5909,0.6148,0.6000,0.5241
RandomForest,0.7521,0.8526,0.6041,0.7864,0.6833,0.4227,0.6001,0.7470
LightGBM,0.7574,0.8526,0.6087,0.7813,0.6843,0.4709,0.6000,0.7297



=== M3: How many minutes does a leg lose at hubs beyond driving time, and which centres cause it?  (winner: LightGBM, n_train=18,947, n_test=7,421)


,MAE,RMSE,MAPE,R2
Ridge,58.30,138.85,90.17,0.1022
RandomForest,49.12,126.02,77.27,0.2604
LightGBM,47.52,122.70,78.00,0.2988
